# DRW Crypto Market Prediction — Analyse Exploratoire (EDA)

**Objectif** : Comprendre la structure des données avant de modéliser.  
**Dataset** : 525 887 lignes × 896 features, résolution minute, mars 2023 – fév. 2024.  
**Target** : `label` — mouvement de prix anonymisé (μ≈0.036, σ≈1.01)

---

## 0. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, normaltest
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
SEED = 42
np.random.seed(SEED)

print('Imports OK')

## 1. Chargement des données

In [ ]:
# Chargement — adapter les chemins si besoin
train = pd.read_parquet('train.parquet')
test  = pd.read_parquet('test.parquet')

print(f'Train : {train.shape[0]:,} lignes × {train.shape[1]} colonnes')
print(f'Test  : {test.shape[0]:,} lignes × {test.shape[1]} colonnes')
print(f'\nColonnes train : {list(train.columns[:10])} ... (+ {train.shape[1]-10} autres)')

In [ ]:
# Identification des features
MARKET_FEATURES = ['bid_qty', 'ask_qty', 'buy_qty', 'sell_qty', 'volume']
X_FEATURES = [c for c in train.columns if c.startswith('X')]
TARGET = 'label'

print(f'Features marché  : {len(MARKET_FEATURES)}')
print(f'Features X (DRW) : {len(X_FEATURES)}')
print(f'Target           : {TARGET}')
print(f'\nPremières features X : {X_FEATURES[:10]}')

## 2. Analyse de la Target (`label`)

In [ ]:
y = train[TARGET]

print('=== Statistiques descriptives de la target ===')
print(f'Moyenne   : {y.mean():.4f}')
print(f'Std       : {y.std():.4f}')
print(f'Min       : {y.min():.4f}')
print(f'Max       : {y.max():.4f}')
print(f'Skewness  : {y.skew():.4f}')
print(f'Kurtosis  : {y.kurt():.4f}')
print(f'NaN       : {y.isna().sum()}')

# Test de normalité
stat, p = normaltest(y.dropna())
print(f'\nTest de normalité (D\'Agostino-Pearson) : stat={stat:.2f}, p={p:.2e}')
if p < 0.05:
    print('→ Distribution NON normale (p < 0.05)')
else:
    print('→ Distribution compatible avec la normalité')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribution de la Target (label)', fontsize=14, fontweight='bold')

# Histogramme
axes[0].hist(y, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Mean={y.mean():.3f}')
axes[0].set_title('Distribution')
axes[0].set_xlabel('label')
axes[0].legend()

# QQ-plot
stats.probplot(y, dist='norm', plot=axes[1])
axes[1].set_title('QQ-Plot vs Normale')

# Série temporelle (sous-échantillon)
axes[2].plot(y.values[:5000], color='steelblue', alpha=0.7, linewidth=0.5)
axes[2].set_title('Série temporelle (5000 premiers points)')
axes[2].set_xlabel('Index')
axes[2].set_ylabel('label')

plt.tight_layout()
plt.show()

In [ ]:
# Autocorrélation de la target
autocorr_lags = [1, 5, 10, 20, 60, 120]
print('=== Autocorrélation de la target ===')
for lag in autocorr_lags:
    ac = y.autocorr(lag=lag)
    print(f'  Lag {lag:4d} : {ac:.4f}')

## 3. Analyse des features marché

In [ ]:
print('=== Corrélations features marché ↔ target ===')
market_corrs = {}
for feat in MARKET_FEATURES:
    corr, p = pearsonr(train[feat].fillna(0), y)
    market_corrs[feat] = corr
    print(f'  {feat:15s} : {corr:+.4f}  (p={p:.2e})')

print(f'\nMax corrélation marché : {max(abs(v) for v in market_corrs.values()):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Features Marché', fontsize=14, fontweight='bold')

# Bar corrélations
colors = ['green' if v > 0 else 'red' for v in market_corrs.values()]
axes[0].bar(market_corrs.keys(), market_corrs.values(), color=colors, alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Corrélation avec la target')
axes[0].set_ylabel('Pearson r')
axes[0].tick_params(axis='x', rotation=30)

# Distribution volume
axes[1].hist(np.log1p(train['volume']), bins=80, color='orange', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribution log(volume + 1)')
axes[1].set_xlabel('log(volume + 1)')

plt.tight_layout()
plt.show()

## 4. Analyse des features X (signaux propriétaires DRW)

In [ ]:
# Calcul corrélations pour toutes les features X
print('Calcul des corrélations pour 890 features X (peut prendre ~1 min)...')

x_corrs = {}
for feat in X_FEATURES:
    corr, _ = pearsonr(train[feat].fillna(0), y)
    if not np.isnan(corr):
        x_corrs[feat] = corr

x_corrs_abs = {k: abs(v) for k, v in x_corrs.items()}
x_corrs_sorted = sorted(x_corrs_abs.items(), key=lambda x: x[1], reverse=True)

print(f'\n=== Top 15 features X par corrélation absolue ===')
for feat, corr in x_corrs_sorted[:15]:
    print(f'  {feat:6s} : {corr:.4f}')

print(f'\nMax corrélation X  : {x_corrs_sorted[0][1]:.4f} ({x_corrs_sorted[0][0]})')
print(f'Mean corrélation X : {np.mean(list(x_corrs_abs.values())):.4f}')
print(f'Features X avec |r| > 0.05 : {sum(1 for v in x_corrs_abs.values() if v > 0.05)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Signal des features X (890 signaux DRW)', fontsize=14, fontweight='bold')

# Distribution des corrélations absolues
corr_values = list(x_corrs_abs.values())
axes[0].hist(corr_values, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(np.mean(corr_values), color='red', linestyle='--',
                label=f'Moyenne = {np.mean(corr_values):.4f}')
axes[0].axvline(0.05, color='orange', linestyle=':', label='Seuil 0.05')
axes[0].set_title('Distribution des |corrélations| avec la target')
axes[0].set_xlabel('|Pearson r|')
axes[0].legend()

# Top 30 features
top30 = x_corrs_sorted[:30]
names, vals = zip(*top30)
axes[1].barh(list(reversed(names)), list(reversed(vals)), color='steelblue', alpha=0.8)
axes[1].set_title('Top 30 features X (|corrélation|)')
axes[1].set_xlabel('|Pearson r|')

plt.tight_layout()
plt.show()

In [ ]:
# Comparaison signal X vs signal marché
max_market = max(abs(v) for v in market_corrs.values())
max_x      = x_corrs_sorted[0][1]
mean_x     = np.mean(corr_values)

print('=== Comparaison Signal Marché vs Signal X ===')
print(f'Max corrélation features marché : {max_market:.4f}')
print(f'Max corrélation features X      : {max_x:.4f}')
print(f'Ratio X / marché                : {max_x / max_market:.1f}x')
print(f'\n→ Les features X ont un signal {max_x / max_market:.1f}x plus fort que les features marché')
print(f'→ Priorité : utiliser les features X dans la modélisation')

## 5. Analyse temporelle & stationnarité

In [ ]:
# Découpage temporel en périodes
n = len(train)
periods = {'Début': train.iloc[:n//4], 
           'Milieu-début': train.iloc[n//4:n//2],
           'Milieu-fin': train.iloc[n//2:3*n//4], 
           'Fin': train.iloc[3*n//4:]}

print('=== Statistiques de la target par période ===')
for name, subset in periods.items():
    y_sub = subset[TARGET]
    print(f'  {name:15s} : mean={y_sub.mean():+.4f}, std={y_sub.std():.4f}, '
          f'min={y_sub.min():.2f}, max={y_sub.max():.2f}')

In [ ]:
# Corrélation des features X par période — instabilité temporelle
best_x_feat = x_corrs_sorted[0][0]  # ex: X21

print(f'=== Corrélation de {best_x_feat} avec target par période ===')
for name, subset in periods.items():
    corr, _ = pearsonr(subset[best_x_feat].fillna(0), subset[TARGET])
    print(f'  {name:15s} : {corr:+.4f}')

In [ ]:
# Volatilité de la target dans le temps
window = 5000
rolling_std  = y.rolling(window).std()
rolling_mean = y.rolling(window).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Dynamique temporelle de la target', fontsize=14, fontweight='bold')

axes[0].plot(rolling_mean, color='steelblue', linewidth=1)
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_title(f'Moyenne glissante (fenêtre={window})')
axes[0].set_ylabel('Mean')

axes[1].plot(rolling_std, color='orange', linewidth=1)
axes[1].set_title(f'Volatilité glissante (fenêtre={window})')
axes[1].set_ylabel('Std')
axes[1].set_xlabel('Index temporel')

plt.tight_layout()
plt.show()

print('→ Changements de régime visibles → forte instabilité temporelle')

## 6. Analyse du shift train/test

In [ ]:
# Comparaison distributions train vs test sur features marché
print('=== Test de Kolmogorov-Smirnov : Train vs Test ===')
aligned = 0
for feat in MARKET_FEATURES:
    stat, p = stats.ks_2samp(train[feat].fillna(0), test[feat].fillna(0))
    aligned_flag =  if p > 0.05 else 
    if p > 0.05:
        aligned += 1
    print(f'  {feat:15s} : KS={stat:.4f}, p={p:.2e} {aligned_flag}')

print(f'\nFeatures alignées train/test : {aligned}/{len(MARKET_FEATURES)}')
print('→ Présence de distribution shift → valider sur les données récentes')

In [ ]:
# Visualisation shift sur une feature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution Shift : Train vs Test', fontsize=14, fontweight='bold')

for i, feat in enumerate(['volume', 'bid_qty']):
    train_vals = np.log1p(train[feat].clip(lower=0))
    test_vals  = np.log1p(test[feat].clip(lower=0))
    axes[i].hist(train_vals, bins=60, alpha=0.5, label='Train', color='steelblue', density=True)
    axes[i].hist(test_vals,  bins=60, alpha=0.5, label='Test',  color='orange',   density=True)
    axes[i].set_title(f'log({feat})')
    axes[i].legend()

plt.tight_layout()
plt.show()

## 7. Valeurs manquantes & qualité des données

In [ ]:
# Analyse des NaN
nan_market = train[MARKET_FEATURES].isna().sum()
nan_x_summary = train[X_FEATURES].isna().sum()

print('=== NaN dans les features marché (train) ===')
print(nan_market.to_string())

print(f'\n=== NaN dans les features X (train) ===')
print(f'Features X sans NaN      : {(nan_x_summary == 0).sum()}')
print(f'Features X avec NaN      : {(nan_x_summary > 0).sum()}')
print(f'Max NaN dans une feature : {nan_x_summary.max()}')
print(f'Total NaN dans target    : {train[TARGET].isna().sum()}')

## 8. Résumé stratégique

In [ ]:
print('=' * 60)
print('RÉSUMÉ STRATÉGIQUE: RECOMMANDATIONS DE MODÉLISATION')
print('=' * 60)

print('''
Bonne pratique:
  - Prioriser les features X (signal 4.3x plus fort que marché)
  - Feature engineering sur les features X sélectionnées
  - Régularisation forte (Ridge/Lasso)
  - Validation time-aware (split 80/20 temporel)
  - Focus sur les données récentes (distribution shift)
  - Sélection des 100 meilleures features par corrélation

 Mauvaise intuition:
  - Architectures neuronales complexes dans des données bruitées
  - Utiliser tout l'historique sans tenir compte du shift
  - Régularisation faible → overfitting
  - Features marché comme signal principal
  - Cross-validation standard (fuite temporelle)

PLAFOND THÉORIQUE DE PERFORMANCE :
  - Signal max : 0.0677 (X21)
  - Corrélation prédite atteignable : ~0.113
  - Objectif réaliste : 0.10 – 0.12
  - Score gagnant : 0.131

CONCLUSION :
  Les modèles simples (Ridge) avec bonne feature engineering
  surpassent les architectures complexes dans ce contexte bruité.
''')